In [ ]:
import pdfplumber
import re
import os
import pandas as pd
from concurrent.futures import ThreadPoolExecutor # Changed to ThreadPool for Jupyter stability

# --- Functions remain the same ---

def extract_spatial_value(words, anchor_texts, x_tolerance=30, index=0, exclude_text=None):
    if isinstance(anchor_texts, str):
        anchor_texts = [anchor_texts]
    candidates = [w for w in words if any(t.upper() in w['text'].upper() for t in anchor_texts)]
    valid_anchors = []
    for cand in candidates:
        if exclude_text:
            preceded_by_excluded = any(
                exclude_text.upper() in w['text'].upper() 
                and abs(w['top'] - cand['top']) < 5
                and 0 < (cand['x0'] - w['x1']) < 80
                for w in words
            )
            if preceded_by_excluded: continue
        if "OPP." in cand['text'].upper() and "OPPONENT" in cand['text'].upper(): continue
        valid_anchors.append(cand)
    if not valid_anchors: return None
    valid_anchors.sort(key=lambda x: (x['top'], x['x0']))
    anchor = valid_anchors[0]
    column_digits = [w for w in words if abs(w['x0'] - anchor['x0']) < x_tolerance and w['top'] > anchor['bottom'] and re.match(r'^\d+$', w['text'])]
    column_digits.sort(key=lambda x: x['top'])
    return column_digits[index]['text'] if len(column_digits) > index else None

def get_header_metric(words, label_text):
    label_anchor = next((w for w in words if label_text.upper() in w['text'].upper()), None)
    if not label_anchor: return None
    candidates = [w for w in words if re.match(r'^\d+$', w['text']) and ((abs(w['top'] - label_anchor['top']) < 10 and w['x0'] > label_anchor['x1']) or (abs(w['x0'] - label_anchor['x0']) < 30 and w['top'] > label_anchor['bottom']))]
    candidates.sort(key=lambda x: (x['top'], x['x0']))
    return candidates[0]['text'] if candidates else None

def get_spatial_team_name(words, page_height):
    header_words = [w for w in words if w['top'] < page_height * 0.12 and not any(x in w['text'].upper() for x in ["THROUGH", "GAMES", "OFFICIAL", "PAGE"]) and not re.match(r'^\d{1,2}/\d{1,2}/\d{2,4}$', w['text'])]
    if not header_words: return None
    header_words.sort(key=lambda x: (x['top'], x['x0']))
    lines = []
    current_line = [header_words[0]]
    for i in range(1, len(header_words)):
        if abs(header_words[i]['top'] - header_words[i-1]['top']) < 3:
            current_line.append(header_words[i])
        else:
            lines.append(" ".join([w['text'] for w in current_line]))
            current_line = [header_words[i]]
    lines.append(" ".join([w['text'] for w in current_line]))
    clean = re.split(r'\(|:|Through Games|\(OFFICIAL\)', lines[0])[0].strip()
    if re.match(r"^(\w\s)+\w$", clean): clean = clean.replace(" ", "")
    return clean

def process_single_pdf(pdf_path):
    all_teams = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            words = page.extract_words()
            full_text = " ".join([w['text'] for w in words])
            team_name = get_spatial_team_name(words, page.height)
            if not team_name: continue
            record_match = re.search(r"(\d{1,2}\s*-\s*\d{1,2})", full_text)
            metrics = {
                "Team": team_name,
                "Record": record_match.group(1).replace(" ", "") if record_match else None,
                "Avg_RPI_Win": get_header_metric(words, "Average RPI Win"),
                "Avg_RPI_Loss": get_header_metric(words, "Average RPI Loss"),
                "RPI_Rank_D1": extract_spatial_value(words, "TEAM", index=0),
                "RPI_Rank_NonConf": extract_spatial_value(words, "TEAM", index=1),
                "SOS_D1": extract_spatial_value(words, ["SUCCESS", "STRENGTH"], index=0, exclude_text="OPP"),
                "SOS_NonConf": extract_spatial_value(words, ["SUCCESS", "STRENGTH"], index=1, exclude_text="OPP"),
                "Opp_SOS_D1": extract_spatial_value(words, "OPP.", index=0),
                "Opp_SOS_NonConf": extract_spatial_value(words, "OPP.", index=1),
            }
            for key, pattern in [("NET", r"NET:\s*(\d+)"), ("KPI", r"KPI:\s*(\d+)"), ("SOR", r"SOR:\s*(\d+)"), ("POM", r"POM:\s*(\d+)"), ("SAG", r"SAG:\s*(\d+)"), ("BPI", r"BPI:\s*(\d+)")]:
                match = re.search(pattern, full_text)
                metrics[key] = match.group(1) if match else None
            all_teams.append(metrics)
    return os.path.splitext(os.path.basename(pdf_path))[0], all_teams

def clean_and_reconcile(year_key, raw_data, base_path):
    df = pd.DataFrame(raw_data)
    cols_to_check = [c for c in df.columns if c != 'Team']
    df = df.dropna(subset=cols_to_check, how='all').reset_index(drop=True)
    
    # Matching the exact naming convention for the reference file
    ref_path = os.path.join(base_path, "team_sheets", f"{year_key}_Team_Sheets_Final.csv")
    if not os.path.exists(ref_path):
        print(f"Warning: Reference CSV not found for {year_key} at {ref_path}")
        return df
    
    ref_df = pd.read_csv(ref_path)
    if len(df) != len(ref_df):
        print(f"Mismatch in {year_key}: Parsed {len(df)} teams, Reference has {len(ref_df)}.")
    
    # Map reference names to parsed data
    df['Team'] = ref_df.iloc[:, 0].values[:len(df)] 
    
    assert not df['Team'].duplicated().any(), f"Duplicate names in {year_key}"
    return df

# --- Main Execution Block for Jupyter ---

pdf_dir = "/Users/michaelharoon/Projects/tasty/march-madness/data/raw/pdf/men/team_sheets/"
base_proj_path = "/Users/michaelharoon/Projects/tasty/march-madness/data/"
output_dir = "/Users/michaelharoon/Projects/tasty/march-madness/data/raw/dirty_csv/test/"

os.makedirs(output_dir, exist_ok=True)
files = [os.path.join(pdf_dir, f) for f in os.listdir(pdf_dir) if f.lower().endswith(".pdf")]

print(f"Processing {len(files)} files...")

# ThreadPoolExecutor is safe for Jupyter/macOS and still parallelizes well here
with ThreadPoolExecutor() as executor:
    results = list(executor.map(process_single_pdf, files))

for year_key, data in results:
    print(f"Cleaning {year_key}...")
    cleaned_df = clean_and_reconcile(year_key, data, base_proj_path)
    save_path = f"/Users/michaelharoon/Projects/tasty/march-madness/data/raw/dirty_csv/test/{year_key}_Cleaned.csv"
    cleaned_df.to_csv(save_path, index=False)
    print(f"Saved: {save_path}")

In [ ]:
import pandas as pd
import os
import re

# Define paths
ref_dir = "/Users/michaelharoon/Projects/tasty/march-madness/data/team_sheets/"
dirty_dir = "/Users/michaelharoon/Projects/tasty/march-madness/data/raw/dirty_csv/test/"

for year in range(2005, 2020):
    # Determine which suffix exists for this year
    base_name = None
    if os.path.exists(os.path.join(ref_dir, f"{year}_Team_Sheets_Selection.csv")):
        base_name = f"{year}_Team_Sheets_Selection"
    elif os.path.exists(os.path.join(ref_dir, f"{year}_Team_Sheets_Final.csv")):
        base_name = f"{year}_Team_Sheets_Final"
    
    if not base_name:
        print(f"Skipping {year}: No reference file (_Selection or _Final) found.")
        continue

    ref_path = os.path.join(ref_dir, f"{base_name}.csv")
    # Assuming the dirty file followed the base_name from the previous parsing step
    dirty_path = os.path.join(dirty_dir, f"{base_name}_Cleaned.csv")
    
    if not os.path.exists(dirty_path):
        print(f"Skipping {year}: Dirty file {base_name}_Cleaned.csv not found.")
        continue
    
    # Load data
    df_ref = pd.read_csv(ref_path)
    df_dirty = pd.read_csv(dirty_path)
    
    # 1. Extract the Date from the dirty 'Team' column
    sample_text = str(df_dirty.iloc[0]['Team'])
    # Matches "March 16, 2019" or "MARCH 6, 2005"
    date_match = re.search(r"March\s+(\d{1,2}),\s+(\d{4})", sample_text, re.IGNORECASE)
    
    if date_match:
        day = date_match.group(1).zfill(2)
        extracted_year = date_match.group(2)
        new_filename = f"{extracted_year}-03-{day}_Team_Sheets.csv"
    else:
        print(f"!! Date match failed for {year}. Falling back to year-only name.")
        new_filename = f"{year}-03-XX_Team_Sheets.csv"

    # 2. Row count verification
    if len(df_ref) != len(df_dirty):
        print(f"!! Mismatch in {year}: Ref {len(df_ref)} rows, Dirty {len(df_dirty)} rows. Skipping.")
        continue
    
    # 3. Overwrite messy names with clean names
    df_dirty['Team'] = df_ref['Team'].values
    
    # 4. Save to final destination
    save_path = f'/Users/michaelharoon/Projects/tasty/march-madness/data/raw/dirty_csv/{new_filename}'
    df_dirty.to_csv(save_path, index=False)
    
    print(f"Processed {year}: {base_name} -> {new_filename}")

print("\nCleanup and renaming complete.")

In [ ]:
import pandas as pd
import os
import xlrd
from openpyxl import load_workbook
print(xlrd.__version__)

def get_unique_columns(directory_path):
    if not os.path.exists(directory_path):
        print(f"Error: The directory {directory_path} does not exist.")
        return set()

    unique_columns = set()
    file_count = 0

    # Iterate through the year-based folders
    for i in range(2003, 2027):
        parent_path = os.path.join(directory_path, f"{i}-team-stats")
        
        # Check if the year folder actually exists before listing
        if not os.path.exists(parent_path):
            continue

        for filename in os.listdir(parent_path):
            file_path = os.path.join(parent_path, filename)
            
            if os.path.isfile(file_path):
                try:
                    # Handle CSV files
                    if filename.endswith('.csv'):
                        df = pd.read_csv(file_path, nrows=0)
                        unique_columns.update(df.columns.tolist())
                        file_count += 1
                    

                    elif filename.endswith('.xls'):
                        try:
                            # Try as true XLS
                            df = pd.read_excel(file_path, engine='xlrd', nrows=0)
                        except Exception:
                            try:
                                # Force openpyxl in read-only mode (bypasses style issues)
                                wb = load_workbook(file_path, read_only=True, data_only=True)
                                ws = wb.active

                                # Extract header row manually
                                header = [cell.value for cell in next(ws.iter_rows(max_row=1))]
                                df = pd.DataFrame(columns=header)

                            except Exception as e:
                                print(f"Could not read {filename}: {e}")
                                continue

                        unique_columns.update(df.columns.tolist())
                        file_count += 1

                except Exception as e:
                    print(f"Could not read {filename}: {e}")

    print(f"Processed {file_count} files (CSV and XLS).")
    return sorted(list(unique_columns))

# --- Execution ---
target_folder = '/Users/michaelharoon/Projects/tasty/march-madness/data'
all_cols = get_unique_columns(target_folder)

print(f"{len(all_cols)} Unique Column Names Found:")
print(60 * "-")
print(*all_cols, sep=",")

In [ ]:
import pathlib
import pyarrow.parquet as pq

base_path = '/Users/michaelharoon/Projects/tasty/march-madness/data/market_data_store'
all_columns = set()

# Recursively find all parquet files
for path in pathlib.Path(base_path).rglob('*.parquet'):
    try:
        # Read only the file metadata/schema
        schema = pq.read_schema(path)
        all_columns.update(schema.names)
    except Exception as e:
        print(f"Could not read {path}: {e}")

print("Distinct Columns:", sorted(list(all_columns)))


In [ ]:
import pandas as pd
import os

def calculate_win_pct(record):
    """Parses 'W-L' string and returns W / (W + L)."""
    try:
        if pd.isna(record) or '-' not in str(record):
            return 0
        wins, losses = map(int, str(record).split('-'))
        if (wins + losses) == 0:
            return 0
        return wins / (wins + losses)
    except (ValueError, ZeroDivisionError):
        return 0

directory_path = '/Users/michaelharoon/Projects/tasty/march-madness/data'

# Process only CSV files in the first level of the directory
for filename in os.listdir(directory_path):
    if filename.endswith(".csv") and filename.startswith("20"):
        file_path = os.path.join(directory_path, filename)
        df = pd.read_csv(file_path)

        df.rename(
            columns={
                'RB_KPI': 'KPI',
                'RB_SOR': 'SOR',
                'PM_BPI': "BPI",
                'PM_POM': "POM",
                'PM_SAG': "SAG"
            },
            inplace=True)

        # Save the updated CSV
        df.to_csv(file_path, index=False)


In [ ]:
import pandas as pd
import os

directory_path = '/Users/michaelharoon/Projects/tasty/march-madness/data/raw/dirty_csv/test'

for filename in os.listdir(directory_path):
    if filename.endswith(".csv") and filename.startswith("20"):
        file_path = os.path.join(directory_path, filename)
        
        try:
            df = pd.read_csv(file_path)
            
            if 'Team' in df.columns:
                # 1. Clean the team names for comparison (strip spaces and uppercase)
                # This ensures "Duke" and "duke " are caught as duplicates
                clean_names = df['Team'].astype(str).str.strip().str.upper()
                
                # 2. Find all instances of duplicates
                # keep=False marks every occurrence of a duplicate as True
                is_duplicate = clean_names.duplicated(keep=False)
                
                if is_duplicate.any():
                    # 3. Filter the original dataframe to show the dupes
                    dupes_found = df[is_duplicate].sort_values(by='Team')
                    
                    print(f"\n[!] {filename}: Found {len(dupes_found)} rows with duplicate team names:")
                    print("-" * 50)
                    # Showing 'Team' and any other relevant columns you might want to see
                    print(dupes_found[['Team']]) 
                    print("-" * 50)
                else:
                    print(f"Clean: {filename}")
                    
        except Exception as e:
            print(f"Error reading {filename}: {e}")

In [ ]:
import pandas as pd
ranks = pd.read_csv('/Users/michaelharoon/Projects/tasty/march-madness/data/kaggle/MMasseyOrdinals.csv')
ranks[ranks['SystemName']=='POM']

In [ ]:
import pandas as pd

# 1. Load Data
ranks = pd.read_csv('/Users/michaelharoon/Projects/tasty/march-madness/data/kaggle/MMasseyOrdinals.csv')
ts_2026 = pd.read_csv('/Users/michaelharoon/Projects/tasty/march-madness/data/team_sheets/2026_Team_Sheet_Final.csv')

# Standardize Massey naming
system_col = 'SystemName' if 'SystemName' in ranks.columns else 'SysteName'

# 2. Identify Massey Historical vs 2026
historical_systems = set(ranks[ranks['Season'] < 2026][system_col].unique())
systems_2026_massey = set(ranks[ranks['Season'] == 2026][system_col].unique())

# Systems that existed before but are missing from 2026 Massey data
dropped_in_2026 = historical_systems - systems_2026_massey

# 3. Analyze 2026 Team Sheet coverage
# Identify columns in Team Sheet that have at least one non-null value
ts_available = [col for col in ts_2026.columns if ts_2026[col].notna().any()]
ts_available_set = set(ts_available)

# Mapping dictionary for known aliases (Team Sheet vs Massey abbreviations)
# Standard Massey codes: SAG (Sagarin), POM (Pomeroy), BPI (ESPN), MOR (Moore), etc.
mapping = {
    'POM': 'POM',
    'SAG': 'SAG',
    'BPI': 'BPI',
    'KPI': 'KPI',
    'SOR': 'SOR',
    'NET_Rank': 'NET',
    'RPI_Rank_D1': 'RPI',
    'PM_T-Rank': 'TRK' # BartTorvik
}

# 4. Logical Comparison
can_fill_dropped = [m for m in dropped_in_2026 if m in ts_available_set or m in mapping.values()]
new_to_2026_ts = ts_available_set - historical_systems

# --- REPORTING ---
print(f"--- MASSEY 2026 GAP ANALYSIS ---")
print(f"Systems active in previous years: {len(historical_systems)}")
print(f"Systems active in 2026 Massey:    {len(systems_2026_massey)}")
print(f"Missing from 2026 Massey:        {len(dropped_in_2026)}")

print(f"\n--- TEAM SHEET RECOVERY POTENTIAL ---")
if can_fill_dropped:
    print(f"Team Sheet can RECOVER these missing Massey systems: {can_fill_dropped}")
else:
    print("Team Sheet does not contain any of the dropped Massey systems.")

print(f"\nNew/Custom features in Team Sheet (not in historical Massey):")
print([c for c in new_to_2026_ts if c not in mapping])

# Detailed check for your "Big Three"
for sys in ['SAG', 'POM', 'BPI']:
    in_massey = sys in systems_2026_massey
    in_ts = sys in ts_available_set
    status = "AVAILABLE" if in_ts or in_massey else "MISSING EVERYWHERE"
    print(f"{sys:3} Status: {status} (Massey: {in_massey}, TeamSheet: {in_ts})")

In [ ]:
import pandas as pd

ords = pd.read_csv('/Users/michaelharoon/Projects/tasty/march-madness/data/kaggle/MMasseyOrdinals.csv')

summary = (
    ords.groupby('SystemName')
    .agg(
        Seasons=('Season', lambda s: sorted(s.unique().tolist())),
        MinDayNum=('RankingDayNum', 'min'),
        MaxDayNum=('RankingDayNum', 'max'),
    )
    .reset_index()
)

for _, row in summary.iterrows():
    if 2026 not in row['Seasons'] or len(row['Seasons']) <=1:
        continue
    print(f"{row['SystemName']}")
    print(f"  Seasons : {row['Seasons']}")
    print(f"  DayNum  : {row['MinDayNum']} – {row['MaxDayNum']}")
    print()

In [ ]:
ords = pd.read_csv('/Users/michaelharoon/Projects/tasty/march-madness/data/kaggle/MMasseyOrdinals.csv')
ords = ords[ords['SystemName']=='WAB']['Season'].unique()
ords

# get KPI, POM,WAB, SOR (get from espn lowkey) from warren nolan

array([2025])

In [ ]:
import pandas as pd
seasons = pd.read_csv('/Users/michaelharoon/Projects/tasty/march-madness/data/kaggle/MSeasons.csv')
print(seasons.columns)
starts = seasons[['Season','DayZero']]
starts

In [ ]:
import pandas as pd
import difflib

_2021 = pd.read_csv('/Users/michaelharoon/Projects/tasty/march-madness/data/team_sheets/2021_Team_Sheet_Final.csv')
_2005 = pd.read_csv('/Users/michaelharoon/Projects/tasty/march-madness/data/team_sheets/2005-04-04_Team_Sheets.csv')

# 1. Exact Matches using Set Intersection
common = set(_2021.columns) & set(_2005.columns)
only_2021 = set(_2021.columns) - set(_2005.columns)
only_2005 = set(_2005.columns) - set(_2021.columns)

# 2. Fuzzy Matching for potential renames/typos
potential_matches = []
for col in only_2021:
    # Finds the closest string in the 2005 list
    match = difflib.get_close_matches(col, list(only_2005), n=1, cutoff=0.7)
    if match:
        potential_matches.append((col, match[0]))

# 3. Print Results
print(f"✅ Common Columns ({len(common)}): {sorted(list(common))}")
print(f"\n🆕 Only in 2021 ({len(only_2021)}): {sorted(list(only_2021))}")
print(f"📜 Only in 2005 ({len(only_2005)}): {sorted(list(only_2005))}")
print(f"\n🔍 Potential Renames (2021 -> 2005): {potential_matches}")


In [143]:
pd.set_option('display.max_rows', None)

bpi = pd.read_csv('/Users/michaelharoon/Projects/tasty/march-madness/data/raw/archived/bpi/espn_bpi_team_rankings_2013_to_present.csv')
bpi = bpi[bpi['season']==2026]

ts = pd.read_csv('/Users/michaelharoon/Projects/tasty/march-madness/data/team_sheets/2026-04-01_Team_Sheet_Partial.csv')

df = pd.merge(bpi, ts, left_on='bpi_rank', right_on='PM_BPI')[['team','Team','PM_BPI','bpi_rank']]
df.rename(columns={'Team':'wn_team', 'team':'espn_team'}, inplace=True)

df.iloc[106]

espn_team    Pacific Tigers
wn_team        Saint Thomas
PM_BPI                  131
bpi_rank                131
Name: 106, dtype: object

In [ ]:
# WN is delayed. espn is up to date
# Map team name ot id then move it to mmassey ordinals
kg_teams = pd.read_csv('/Users/michaelharoon/Projects/tasty/march-madness/data/kaggle/MTeams.csv')


,espn_team,wn_team,BPI,bpi_rank
5,Illinois Fighting Illini,Iowa State,6,6
6,Iowa State Cyclones,Illinois,7,7
7,Purdue Boilermakers,Gonzaga,8,8
8,Gonzaga Bulldogs,Purdue,9,9
9,UConn Huskies,Connecticut,10,10
